[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/limaraujo/Projeto-ANS/blob/main/codigo/Projeto_AMNS_Equipe8.ipynb)

# Projeto AM-NS 2026-2 — Equipe 8

Notebook completo de agrupamento e redução de dimensionalidade na base Iris.

Todo o código dos experimentos está nas células deste notebook. A base é carregada diretamente de sklearn.datasets, disponível no Google Colab.

## Preparação

Não é necessário clonar o GitHub, enviar ZIP, montar o Google Drive ou fornecer arquivos Python. O notebook contém as implementações completas.

A única fonte de dados é sklearn.datasets.load_iris(), já disponível no ambiente do Colab. Execute em ordem com **Ambiente de execução → Executar tudo**.

> A Questão 1 realiza 1.144 ajustes piloto e 252 ajustes oficiais; sua execução pode levar alguns minutos.

In [ ]:
import io
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

NO_COLAB = "google.colab" in sys.modules
if NO_COLAB:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            "fuzzy-c-means==2.3.1", "kmedoids==0.5.5",
        ],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "umap-learn"],
        check=True,
    )
    # Evita o carregamento da integração opcional do UMAP com TensorFlow.
    sys.modules["tensorflow"] = None

area_execucao = TemporaryDirectory(prefix="projeto-amns-colab-")
raiz_execucao = Path(area_execucao.name)
pasta_codigo = raiz_execucao / "codigo"
(pasta_codigo / "iris").mkdir(parents=True, exist_ok=True)
print(f"Área temporária da execução: {raiz_execucao}")


## Dataset do Colab

A célula abaixo obtém a Iris por load_iris() e prepara, apenas durante a execução, o formato consumido pelas rotinas do projeto.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris_colab = load_iris()
nomes_especies = [f"Iris-{nome}" for nome in iris_colab.target_names]
tabela_iris = pd.DataFrame(iris_colab.data)
tabela_iris[4] = [nomes_especies[indice] for indice in iris_colab.target]
caminho_iris = pasta_codigo / "iris" / "iris.data"
tabela_iris.to_csv(caminho_iris, header=False, index=False)

print("Fonte: sklearn.datasets.load_iris")
print(f"Amostras: {len(tabela_iris)} | Atributos: {iris_colab.data.shape[1]}")
tabela_iris.head()


## Código completo dos algoritmos

Implementações e adaptadores dos sete agrupadores, métricas, gráficos e persistência.

In [ ]:
# Algoritmos, métricas, gráficos e persistência.
__file__ = str(pasta_codigo / "algoritmos.py")
"""Algoritmos, métricas e persistência usados nos experimentos da base Iris."""

import json
from collections.abc import Iterable
from pathlib import Path
from typing import Any, Literal

import matplotlib
import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from fcmeans import FCM, KFCM
from kmedoids import pam
from scipy.cluster.hierarchy import cut_tree, dendrogram, linkage
from scipy.spatial.distance import cdist, pdist
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler


class ResultadoAgrupamento(BaseModel):
    """Representa o resultado de um algoritmo de agrupamento."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    algoritmo: str
    tipo: Literal["crisp", "fuzzy", "probabilistico", "hierarquico", "densidade"]
    rotulos: np.ndarray
    objetivo: float | None = None
    parametros: dict[str, Any] = Field(default_factory=dict)
    centros: np.ndarray | None = None
    pertinencias: np.ndarray | None = None  # Fuzzy K-Means
    probabilidades: np.ndarray | None = None  # GMM/EM
    hierarquia: np.ndarray | None = None  # Matriz de ligações para o dendrograma
    ruido: int = 0


experimental_algorithms: dict[str, dict[str, Any]] = {}
CAMINHO_IRIS_PADRAO = Path(__file__).with_name("iris") / "iris.data"


def algoritmo(
    nome: str,
    execucoes: int = 10,
    objetivo: Literal["minimizar", "maximizar"] = "minimizar",
):
    """Registra uma função de agrupamento para execução pelo seu nome."""

    def decorator(func):
        experimental_algorithms[nome] = {
            "funcao": func,
            "execucoes": execucoes,
            "objetivo": objetivo,
        }
        return func

    return decorator


def executar_algoritmo(nome: str, *args: Any, **kwargs: Any) -> Any:
    """Executa o algoritmo registrado em ``nome``."""
    try:
        funcao = experimental_algorithms[nome]["funcao"]
    except KeyError as erro:
        raise ValueError(f"Algoritmo desconhecido: {nome}") from erro
    return funcao(*args, **kwargs)


def executar_melhores_por_k(
    nome: str,
    data: np.ndarray,
    valores_k: Iterable[int] = range(1, 6),
    seed_inicial: int = 42,
    **parametros_algoritmo: Any,
) -> dict[int, ResultadoAgrupamento]:
    """Executa um algoritmo particional e retém o melhor resultado de cada ``k``.

    O número de inicializações e o sentido de otimização são obtidos do
    registro do algoritmo. As sementes são consecutivas, a partir de
    ``seed_inicial``.
    """
    todas_execucoes = executar_todas_por_k(
        nome, data, valores_k, seed_inicial, **parametros_algoritmo
    )
    return selecionar_melhores_por_k(nome, todas_execucoes)


def executar_todas_por_k(
    nome: str,
    data: np.ndarray,
    valores_k: Iterable[int] = range(1, 6),
    seed_inicial: int = 42,
    **parametros_algoritmo: Any,
) -> dict[int, list[ResultadoAgrupamento]]:
    """Executa as dez inicializações de cada ``k`` com os parâmetros fornecidos."""
    try:
        quantidade_execucoes = experimental_algorithms[nome]["execucoes"]
    except KeyError as erro:
        raise ValueError(f"Algoritmo desconhecido: {nome}") from erro
    if quantidade_execucoes <= 1:
        raise ValueError(f"{nome} não é um algoritmo particional com reinicializações.")

    return {
        k: [
            executar_algoritmo(
                nome, data, k=k, seed=seed_inicial + tentativa, **parametros_algoritmo
            )
            for tentativa in range(quantidade_execucoes)
        ]
        for k in valores_k
    }


def selecionar_melhores_por_k(
    nome: str,
    todas_execucoes: dict[int, list[ResultadoAgrupamento]],
) -> dict[int, ResultadoAgrupamento]:
    """Seleciona a melhor inicialização de cada ``k`` pela função objetivo."""
    try:
        registro = experimental_algorithms[nome]
    except KeyError as erro:
        raise ValueError(f"Algoritmo desconhecido: {nome}") from erro

    seletor = max if registro["objetivo"] == "maximizar" else min
    melhores: dict[int, ResultadoAgrupamento] = {}
    for k, resultados in todas_execucoes.items():
        if not resultados or any(resultado.objetivo is None for resultado in resultados):
            raise ValueError(f"{nome} não fornece função objetivo para seleção.")
        melhores[k] = seletor(resultados, key=lambda resultado: resultado.objetivo)
    return melhores


def calcular_silhueta(
    data: np.ndarray,
    rotulos: np.ndarray,
    ignorar_ruido: bool = False,
) -> float:
    """Calcula a silhueta média de uma partição crisp.

    Retorna ``np.nan`` quando a métrica não é definida, como no caso de um
    único cluster. Para DBSCAN, use ``ignorar_ruido=True`` para remover os
    pontos de rótulo ``-1`` antes do cálculo.
    """
    X = np.asarray(data, dtype=float)
    labels = np.asarray(rotulos)

    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
        raise ValueError("data deve ser uma matriz bidimensional não vazia.")
    if not np.isfinite(X).all():
        raise ValueError("data não pode conter NaN ou valores infinitos.")
    if labels.ndim != 1 or len(labels) != len(X):
        raise ValueError("rotulos deve ter um rótulo para cada amostra de data.")

    if ignorar_ruido:
        mascara = labels != -1
        X, labels = X[mascara], labels[mascara]

    quantidade_clusters = len(np.unique(labels))
    if len(X) < 2 or quantidade_clusters < 2 or quantidade_clusters >= len(X):
        return float("nan")

    return float(silhouette_score(X, labels, metric="euclidean"))


def selecionar_k_por_silhueta(
    data: np.ndarray,
    resultados: dict[int, ResultadoAgrupamento],
) -> tuple[int, dict[int, float]]:
    """Calcula Sil(k) e retorna o ``k*`` com a maior silhueta válida."""
    silhuetas = {
        k: calcular_silhueta(data, resultado.rotulos) for k, resultado in resultados.items()
    }
    valores_validos = {k: sil for k, sil in silhuetas.items() if not np.isnan(sil)}
    if not valores_validos:
        raise ValueError("Nenhuma partição possui silhueta definida.")

    k_estrela = max(valores_validos, key=valores_validos.get)
    return k_estrela, silhuetas


def calcular_ari(
    rotulos_verdadeiros: np.ndarray,
    rotulos_preditos: np.ndarray,
    ignorar_ruido: bool = False,
) -> float:
    """Calcula o Índice de Rand Ajustado (ARI) entre espécies e clusters.

    Com ``ignorar_ruido=True``, remove de ambos os vetores as observações com
    rótulo previsto ``-1`` e retorna NaN quando restam menos de duas observações.
    """
    y_true = np.asarray(rotulos_verdadeiros)
    y_pred = np.asarray(rotulos_preditos)
    if y_true.ndim != 1 or y_pred.ndim != 1 or len(y_true) != len(y_pred):
        raise ValueError(
            "Os dois vetores de rótulos devem ser unidimensionais e ter o mesmo tamanho."
        )
    if len(y_true) == 0:
        raise ValueError("Os vetores de rótulos não podem ser vazios.")
    if ignorar_ruido:
        mascara = y_pred != -1
        if np.count_nonzero(mascara) < 2:
            return float("nan")
        y_true, y_pred = y_true[mascara], y_pred[mascara]
    return float(adjusted_rand_score(y_true, y_pred))


def condensar_resultados(
    data: np.ndarray,
    rotulos_verdadeiros: np.ndarray,
    resultados_por_algoritmo: dict[str, dict[int, ResultadoAgrupamento]],
    resultado_dbscan: ResultadoAgrupamento | None = None,
) -> pd.DataFrame:
    """Cria uma tabela comparativa dos resultados selecionados.

    Para métodos com ``k``, a linha usa a partição de maior silhueta (``k*``).
    Para DBSCAN, as duas métricas consideram apenas os pontos atribuídos a grupos.
    """
    linhas: list[dict[str, Any]] = []
    for nome, resultados in resultados_por_algoritmo.items():
        k_estrela, silhuetas = selecionar_k_por_silhueta(data, resultados)
        resultado = resultados[k_estrela]
        linhas.append(
            {
                "Algoritmo": nome,
                "k*": k_estrela,
                "Silhueta": silhuetas[k_estrela],
                "ARI": calcular_ari(rotulos_verdadeiros, resultado.rotulos),
                "Clusters": len(np.unique(resultado.rotulos)),
                "Ruído": 0,
                "Ruído (%)": 0.0,
                "Amostras avaliadas": len(resultado.rotulos),
                "Cobertura (%)": 100.0,
            }
        )

    if resultado_dbscan is not None:
        rotulos = resultado_dbscan.rotulos
        quantidade_clusters = len(set(rotulos) - {-1})
        quantidade_ruido = int(np.sum(rotulos == -1))
        quantidade_avaliadas = len(rotulos) - quantidade_ruido
        linhas.append(
            {
                "Algoritmo": "dbscan",
                "k*": "automático",
                "Silhueta": calcular_silhueta(data, rotulos, ignorar_ruido=True),
                "ARI": calcular_ari(rotulos_verdadeiros, rotulos, ignorar_ruido=True),
                "Clusters": quantidade_clusters,
                "Ruído": quantidade_ruido,
                "Ruído (%)": 100 * quantidade_ruido / len(rotulos),
                "Amostras avaliadas": quantidade_avaliadas,
                "Cobertura (%)": 100 * quantidade_avaliadas / len(rotulos),
            }
        )

    return pd.DataFrame(linhas)


def plotar_silhuetas(
    nome_algoritmo: str,
    silhuetas: dict[int, float],
    k_estrela: int,
    diretorio_saida: str | Path = "figuras",
) -> Path:
    """Salva, com Seaborn, o gráfico Sil × k de um algoritmo.

    A silhueta de ``k=1`` não é definida e por isso é marcada no gráfico,
    mas não entra na linha nem na escolha de ``k*``.
    """
    diretorio = Path(diretorio_saida)
    diretorio.mkdir(parents=True, exist_ok=True)

    ks = sorted(silhuetas)
    ks_validos = [k for k in ks if not np.isnan(silhuetas[k])]
    valores_validos = [silhuetas[k] for k in ks_validos]

    sns.set_theme(style="whitegrid", context="notebook")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.lineplot(
        x=ks_validos,
        y=valores_validos,
        marker="o",
        linewidth=2,
        color="#2676a1",
        ax=ax,
    )
    ax.axvline(k_estrela, color="#c43c39", linestyle="--", label=f"k* = {k_estrela}")
    ax.scatter(k_estrela, silhuetas[k_estrela], color="#c43c39", s=70, zorder=3)

    ax.set(
        title=f"Silhueta × k — {nome_algoritmo}",
        xlabel="Número de clusters (k)",
        ylabel="Silhueta média",
    )
    ax.set_xticks(
        ks,
        labels=[f"{k}\n(indefinida)" if np.isnan(silhuetas[k]) else str(k) for k in ks],
    )
    ax.legend()
    fig.tight_layout()

    nome_arquivo = f"silhueta_{nome_algoritmo.lower().replace(' ', '_').replace('-', '_')}.png"
    caminho_saida = diretorio / nome_arquivo
    fig.savefig(caminho_saida, dpi=150)
    plt.close(fig)
    return caminho_saida


def carregar_iris(
    caminho: str | Path | None = None,
    padronizar: bool = True,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    """Carrega a Iris local, separa ``X`` e ``y`` e opcionalmente padroniza X."""
    caminho = CAMINHO_IRIS_PADRAO if caminho is None else Path(caminho)
    colunas = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
    dados = pd.read_csv(caminho, header=None, names=colunas).dropna(how="all")
    if dados.shape[0] != 150:
        raise ValueError(f"A base Iris deve conter 150 observações; foram lidas {dados.shape[0]}.")

    X = dados[colunas[:-1]].to_numpy(dtype=float)
    y = dados["species"].to_numpy()
    configuracao: dict[str, Any] = {
        "padronizado": padronizar,
        "atributos": colunas[:-1],
    }
    if padronizar:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
        configuracao["media_original"] = scaler.mean_.tolist()
        configuracao["escala_original"] = scaler.scale_.tolist()
    return X, y, configuracao


def plotar_dendrograma(
    resultado_upgma: ResultadoAgrupamento,
    diretorio_saida: str | Path = "figuras",
) -> Path:
    """Salva o dendrograma UPGMA Manhattan com estilo visual do Seaborn."""
    if resultado_upgma.hierarquia is None:
        raise ValueError("O resultado não contém uma matriz de ligações hierárquicas.")
    diretorio = Path(diretorio_saida)
    diretorio.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", context="notebook")
    fig, ax = plt.subplots(figsize=(12, 5))
    dendrogram(resultado_upgma.hierarquia, no_labels=True, ax=ax, color_threshold=0)
    ax.set(
        title="Dendrograma — UPGMA (Manhattan / city-block)",
        xlabel="Observações",
        ylabel="Distância",
    )
    fig.tight_layout()
    caminho_saida = diretorio / "dendrograma_upgma.png"
    fig.savefig(caminho_saida, dpi=150)
    plt.close(fig)
    return caminho_saida


def _para_json(valor: Any) -> Any:
    """Converte arrays e escalares NumPy em valores serializáveis em JSON."""
    if isinstance(valor, BaseModel):
        return _para_json(valor.model_dump())
    if isinstance(valor, np.ndarray):
        return _para_json(valor.tolist())
    if isinstance(valor, np.generic):
        return _para_json(valor.item())
    if isinstance(valor, dict):
        return {str(chave): _para_json(item) for chave, item in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [_para_json(item) for item in valor]
    if isinstance(valor, float) and not np.isfinite(valor):
        return None
    return valor


def salvar_json(caminho: str | Path, conteudo: Any) -> Path:
    """Persiste resultados estruturados de forma legível e reprodutível."""
    destino = Path(caminho)
    destino.parent.mkdir(parents=True, exist_ok=True)
    with destino.open("w", encoding="utf-8") as arquivo:
        json.dump(_para_json(conteudo), arquivo, ensure_ascii=False, indent=2, allow_nan=False)
    return destino


def _validar_entrada(data: np.ndarray, k: int) -> np.ndarray:
    """Converte os dados para matriz numérica e valida ``k``."""
    X = np.asarray(data, dtype=float)
    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
        raise ValueError("data deve ser uma matriz bidimensional não vazia.")
    if not np.isfinite(X).all():
        raise ValueError("data não pode conter NaN ou valores infinitos.")
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= X.shape[0]:
        raise ValueError("k deve ser um inteiro entre 1 e o número de amostras.")
    return X


@algoritmo("k-means")
def k_means(
    data: np.ndarray, k: int, seed: int, *, init: str = "random"
) -> ResultadoAgrupamento:
    """Agrupa os dados com K-Means e retorna os rótulos e centros encontrados."""
    X = _validar_entrada(data, k)
    modelo = KMeans(
        n_clusters=k,
        n_init=1,
        init=init,
        random_state=seed,
    ).fit(X)
    return ResultadoAgrupamento(
        algoritmo="k-means",
        tipo="crisp",
        rotulos=modelo.labels_.copy(),
        objetivo=float(modelo.inertia_),
        parametros={
            "k": k,
            "seed": seed,
            "n_init": 1,
            "init": init,
            "n_iter": modelo.n_iter_,
        },
        centros=modelo.cluster_centers_.copy(),
    )


def _executar_fuzzy(
    data: np.ndarray, k: int, seed: int, m: float, gamma: float | None = None
) -> ResultadoAgrupamento:
    """Compartilha ajuste e conversão crisp dos dois métodos fuzzy."""
    X = _validar_entrada(data, k)
    if not np.isfinite(m) or m <= 1:
        raise ValueError("m deve ser finito e maior que 1.")
    parametros = dict(k=k, seed=seed, m=m, max_iter=300, error=1e-5, n_init=1)
    opcoes = dict(
        n_clusters=k,
        m=m,
        max_iter=parametros["max_iter"],
        error=parametros["error"],
        n_init=parametros["n_init"],
        random_state=seed,
    )
    modelo = FCM(**opcoes) if gamma is None else KFCM(**opcoes, gamma=gamma)
    modelo.fit(X)
    centros, pertinencias = modelo.centers.copy(), modelo.u.copy()
    distancias = np.sum((X[:, None, :] - centros[None, :, :]) ** 2, axis=2)
    nome = "fuzzy-k-means"
    if gamma is not None:
        nome = "kernel-fuzzy-k-means"
        parametros.update(gamma=float(modelo.gamma_), kernel="rbf")
        distancias = -2 * np.expm1(-parametros["gamma"] * distancias)
    return ResultadoAgrupamento(
        algoritmo=nome,
        tipo="fuzzy",
        rotulos=np.argmax(pertinencias, axis=1),
        objetivo=float(np.sum(pertinencias**m * distancias)),
        parametros=parametros,
        centros=centros,
        pertinencias=pertinencias,
    )


@algoritmo("fuzzy-k-means")
def fuzzy_k_means(
    data: np.ndarray, k: int, seed: int, *, m: float = 2.0
) -> ResultadoAgrupamento:
    """Executa uma inicialização do Fuzzy C-Means."""
    return _executar_fuzzy(data, k, seed, m)


@algoritmo("kernel-fuzzy-k-means")
def kernel_fuzzy_k_means(
    data: np.ndarray, k: int, seed: int, *, m: float = 2.0, gamma: float = 0.25
) -> ResultadoAgrupamento:
    """Executa uma inicialização do Kernel Fuzzy C-Means com kernel RBF."""
    if not np.isfinite(gamma) or gamma <= 0:
        raise ValueError("gamma deve ser finito e positivo.")
    return _executar_fuzzy(data, k, seed, m, gamma)


@algoritmo("k-medoids")
def kmedoids(
    data: np.ndarray, k: int, seed: int, *, init: str = "random"
) -> ResultadoAgrupamento:
    """Executa PAM com a inicialização escolhida e distância euclidiana."""

    X = _validar_entrada(data, k)
    distancias = cdist(X, X, metric="euclidean")
    # Para um único grupo, a busca exaustiva encontra o medoide ótimo.
    n_iter = None
    if k == 1:
        indices_medoides = np.array([np.argmin(distancias.sum(axis=0))])
        rotulos = np.zeros(len(X), dtype=int)
        objetivo = float(distancias[:, indices_medoides[0]].sum())
    else:
        modelo = pam(
            distancias,
            k,
            init=init,
            max_iter=300,
            random_state=seed,
        )
        indices_medoides = np.asarray(modelo.medoids, dtype=int)
        rotulos = modelo.labels.copy()
        objetivo = float(modelo.loss)
        n_iter = modelo.n_iter

    centros = X[indices_medoides].copy()

    return ResultadoAgrupamento(
        algoritmo="k-medoids",
        tipo="crisp",
        rotulos=rotulos,
        objetivo=objetivo,
        parametros={
            "k": k,
            "seed": seed,
            "metric": "euclidean",
            "method": "pam",
            "init": init if k > 1 else "exhaustive",
            "max_iter": 300 if k > 1 else None,
            "n_iter": n_iter,
            "medoid_indices": indices_medoides.tolist(),
        },
        centros=centros,
    )


@algoritmo("gaussian-mixture", objetivo="maximizar")
def gaussian_mixture(
    data: np.ndarray,
    k: int,
    seed: int,
    *,
    covariance_type: str = "full",
    reg_covar: float = 1e-6,
) -> ResultadoAgrupamento:
    """Agrupa os dados com Gaussian Mixture Model (GMM) e retorna os rótulos e centros encontrados."""
    X = _validar_entrada(data, k)
    modelo = GaussianMixture(
        n_components=k,
        covariance_type=covariance_type,
        random_state=seed,
        tol=1e-5,
        reg_covar=reg_covar,
        n_init=1,
        init_params="random_from_data",
    ).fit(X)

    probabilidades = modelo.predict_proba(X)
    objetivo = float(modelo.score(X) * len(X))

    return ResultadoAgrupamento(
        algoritmo="gaussian-mixture",
        tipo="probabilistico",
        rotulos=np.argmax(probabilidades, axis=1),
        objetivo=objetivo,
        parametros={
            "k": k,
            "seed": seed,
            "covariance_type": covariance_type,
            "tol": 1e-5,
            "reg_covar": reg_covar,
            "n_init": 1,
            "init_params": "random_from_data",
            "converged": modelo.converged_,
            "n_iter": modelo.n_iter_,
        },
        # ``means_`` é exposto pelos stubs do scikit-learn como array-like;
        # ``np.array`` materializa uma cópia concreta e preserva o isolamento
        # do resultado em relação ao estimador ajustado.
        centros=np.array(modelo.means_, copy=True),
        probabilidades=probabilidades,
    )


@algoritmo("upgma", execucoes=1)
def upgma(
    data: np.ndarray, valores_k: tuple[int, ...] = (1, 2, 3, 4, 5)
) -> dict[int, ResultadoAgrupamento]:
    """Constrói uma hierarquia Manhattan e extrai todos os cortes solicitados.

    Cada resultado inclui a mesma matriz de ligações, utilizável por
    ``scipy.cluster.hierarchy.dendrogram``.
    """
    X = _validar_entrada(data, 1)
    if len(X) < 2:
        raise ValueError("UPGMA requer pelo menos duas amostras.")
    if not valores_k:
        raise ValueError("valores_k deve conter ao menos um valor.")
    for k in valores_k:
        _validar_entrada(X, k)
    hierarquia = linkage(pdist(X, metric="cityblock"), method="average")
    cortes = cut_tree(hierarquia, n_clusters=list(valores_k))
    return {
        k: ResultadoAgrupamento(
            algoritmo="upgma",
            tipo="hierarquico",
            rotulos=cortes[:, i].copy(),
            parametros={"k": k, "metric": "manhattan", "linkage": "average"},
            hierarquia=hierarquia.copy(),
        )
        for i, k in enumerate(valores_k)
    }


@algoritmo("dbscan", execucoes=1)
def dbscan(
    data: np.ndarray,
    eps: float = 0.5,
    min_samples: int = 5,
) -> ResultadoAgrupamento:

    X = np.asarray(data, dtype=float)

    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
        raise ValueError("data deve ser uma matriz bidimensional não vazia.")

    if not np.isfinite(X).all():
        raise ValueError("data não pode conter NaN ou valores infinitos.")

    modelo = DBSCAN(
        eps=eps,
        min_samples=min_samples,
        metric="euclidean",
    )

    rotulos = modelo.fit_predict(X)

    ruido = int(np.sum(rotulos == -1))

    return ResultadoAgrupamento(
        algoritmo="dbscan",
        tipo="densidade",
        rotulos=rotulos.copy(),
        objetivo=None,
        parametros={
            "eps": eps,
            "min_samples": min_samples,
            "metric": "euclidean",
        },
        ruido=ruido,
    )


## Código completo da tunagem

Grades de hiperparâmetros, critérios de seleção e análises de sensibilidade.

In [ ]:
# Busca e análise de hiperparâmetros.
# Reutiliza diretamente os símbolos da célula anterior.
__file__ = str(pasta_codigo / "tunagem.py")
"""Busca dos hiperparâmetros de agrupamento pelo protocolo da Questão 1."""

from dataclasses import dataclass
from itertools import product
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from threadpoolctl import threadpool_limits


SEMENTES_PILOTO = range(42, 52)
VALORES_K = range(2, 6)
COBERTURA_MINIMA_DBSCAN = 95.0
PADRAO_DBSCAN = (0.8, 5)

NOMES_METODOS = {
    "k-means": "K-Means",
    "fuzzy-k-means": "Fuzzy K-Means",
    "kernel-fuzzy-k-means": "Kernel Fuzzy K-Means",
    "k-medoids": "K-Medoids",
    "gaussian-mixture": "GMM/EM",
}


@dataclass
class ResultadoBusca:
    particionais: pd.DataFrame
    dbscan: pd.DataFrame
    execucoes: dict[str, Any]
    metadados: dict[str, Any]


def grade_particionais() -> dict[str, list[dict[str, Any]]]:
    """Mantém o perfil de referência na primeira posição de cada grade."""
    valores_m = (2.0, 1.5, 2.5)
    return {
        "k-means": [
            {"init": inicializacao} for inicializacao in ("random", "k-means++")
        ],
        "fuzzy-k-means": [{"m": m} for m in valores_m],
        "kernel-fuzzy-k-means": [
            {"m": m, "gamma": gamma}
            for m, gamma in product(valores_m, (0.25, 0.05, 1.0))
        ],
        "k-medoids": [{"init": inicializacao} for inicializacao in ("random", "build")],
        "gaussian-mixture": [
            {"covariance_type": tipo, "reg_covar": regularizacao}
            for tipo, regularizacao in product(
                ("full", "tied", "diag", "spherical"), (1e-6, 1e-4, 1e-2)
            )
        ],
    }


def grade_dbscan(dados: np.ndarray) -> tuple[float, list[tuple]]:
    """Usa raios absolutos e relativos à mediana da distância ao quinto vizinho."""
    distancias, _ = NearestNeighbors(n_neighbors=5).fit(dados).kneighbors(dados)
    referencia = float(np.median(distancias[:, -1]))
    configuracoes = [
        ("eps_fixo", eps, min_samples, None)
        for eps, min_samples in product((0.4, 0.6, 0.8, 1.0), (3, 5, 10))
    ]
    configuracoes.extend(
        ("eps_relativo", referencia * fator, min_samples, fator)
        for fator, min_samples in product((0.75, 1.0, 1.5, 2.0), (3, 5, 10))
    )
    return referencia, configuracoes


def _ajustar_perfil(nome, dados, parametros):
    execucoes = {k: [] for k in VALORES_K}
    registros = {k: [] for k in VALORES_K}
    for k, seed in product(VALORES_K, SEMENTES_PILOTO):
        registro = dict(
            seed=seed,
            objetivo=None,
            rotulos=None,
            convergiu=None,
            iteracoes=None,
            erro=None,
        )
        try:
            resultado = executar_algoritmo(nome, dados, k=k, seed=seed, **parametros)
            if resultado.objetivo is None or not np.isfinite(resultado.objetivo):
                raise ValueError("Função objetivo indefinida ou não finita.")
        except (ValueError, FloatingPointError, np.linalg.LinAlgError) as erro:
            registro["erro"] = str(erro)
        else:
            execucoes[k].append(resultado)
            registro.update(
                objetivo=resultado.objetivo,
                rotulos=resultado.rotulos,
                convergiu=resultado.parametros.get("converged"),
                iteracoes=resultado.parametros.get("n_iter"),
            )
        registros[k].append(registro)

    melhores = selecionar_melhores_por_k(
        nome, {k: ajustes for k, ajustes in execucoes.items() if ajustes}
    )
    return execucoes, registros, melhores


def _buscar_particionais(dados):
    selecionados, registros = {}, {}
    linhas = []
    for nome, perfis in grade_particionais().items():
        print(f"Testando {len(perfis)} perfis de {NOMES_METODOS[nome]}...")
        registros[nome] = []
        medias = {}
        linhas_perfis = {}
        for configuracao, parametros in enumerate(perfis, start=1):
            execucoes, registros_perfil, melhores = _ajustar_perfil(
                nome, dados, parametros
            )
            registros[nome].append(
                {
                    "configuracao": configuracao,
                    "parametros": parametros,
                    "referencia": configuracao == 1,
                    "execucoes": registros_perfil,
                }
            )
            linhas_perfil = []
            for k in VALORES_K:
                melhor = melhores.get(k)
                convergencias = [
                    r["convergiu"]
                    for r in registros_perfil[k]
                    if r["convergiu"] is not None
                ]
                linhas_perfil.append(
                    {
                        "algoritmo": nome,
                        "configuracao": configuracao,
                        "k": k,
                        **parametros,
                        "seed": melhor.parametros["seed"] if melhor else None,
                        "objetivo": melhor.objetivo if melhor else None,
                        "grupos": len(np.unique(melhor.rotulos)) if melhor else None,
                        "silhueta": (
                            calcular_silhueta(dados, melhor.rotulos)
                            if melhor
                            else np.nan
                        ),
                        "convergidas": sum(convergencias) if convergencias else None,
                        "falhas": len(SEMENTES_PILOTO) - len(execucoes[k]),
                        "referencia": configuracao == 1,
                    }
                )
            linhas_perfis[configuracao] = linhas_perfil
            silhuetas = [linha["silhueta"] for linha in linhas_perfil]
            if np.isfinite(silhuetas).all():
                medias[configuracao] = float(np.mean(silhuetas))

        if not medias:
            raise ValueError(
                f"Nenhum perfil de {nome} tem silhuetas válidas em todos os k."
            )
        # A primeira configuração na ordem da grade é a referência.
        configuracao = max(medias, key=medias.__getitem__)
        selecionados[nome] = {
            "configuracao": configuracao,
            "parametros": perfis[configuracao - 1],
            "silhueta_media": medias[configuracao],
        }
        for id_perfil, linhas_perfil in linhas_perfis.items():
            for linha in linhas_perfil:
                linha.update(
                    silhueta_media=medias.get(id_perfil, np.nan),
                    configuracao_selecionada=id_perfil == configuracao,
                )
            linhas.extend(linhas_perfil)
    return selecionados, registros, pd.DataFrame(linhas)


def _buscar_dbscan(dados, parametros_manuais):
    referencia, configuracoes = grade_dbscan(dados)
    if parametros_manuais and all(
        (eps, min_samples) != parametros_manuais
        for _, eps, min_samples, _ in configuracoes
    ):
        configuracoes.append(("manual", *parametros_manuais, None))

    resultados, linhas = [], []
    for configuracao, (regra, eps, min_samples, fator) in enumerate(
        configuracoes, start=1
    ):
        resultado = dbscan(dados, eps=eps, min_samples=min_samples)
        resultados.append(resultado)
        retidos = len(dados) - resultado.ruido
        cobertura = 100 * retidos / len(dados)
        silhueta = calcular_silhueta(dados, resultado.rotulos, ignorar_ruido=True)
        linhas.append(
            {
                "configuracao": configuracao,
                "regra": regra,
                "eps": eps,
                "min_samples": min_samples,
                "fator": fator,
                "referencia_5_vizinhos": referencia,
                "grupos": len(np.unique(resultado.rotulos[resultado.rotulos != -1])),
                "ruido": resultado.ruido,
                "amostras_avaliadas": retidos,
                "cobertura_pct": cobertura,
                "silhueta": silhueta,
                "elegivel": bool(
                    np.isfinite(silhueta) and cobertura >= COBERTURA_MINIMA_DBSCAN
                ),
            }
        )
    if parametros_manuais:
        melhor = next(
            linha
            for linha in linhas
            if (linha["eps"], linha["min_samples"]) == parametros_manuais
        )
    else:
        elegiveis = [linha for linha in linhas if linha["elegivel"]]
        if not elegiveis:
            raise ValueError(
                "A grade DBSCAN não encontrou uma silhueta válida com cobertura de "
                f"pelo menos {COBERTURA_MINIMA_DBSCAN:g}%."
            )
        melhor = max(
            elegiveis,
            key=lambda linha: (
                linha["silhueta"],
                linha["cobertura_pct"],
                -linha["configuracao"],
            ),
        )

    for linha in linhas:
        linha["selecionado"] = linha is melhor
    registros = [
        {"configuracao": linha["configuracao"], "rotulos": resultado.rotulos}
        for linha, resultado in zip(linhas, resultados, strict=True)
    ]
    return melhor, registros, pd.DataFrame(linhas)


def comparar_amostra_dbscan(
    dados: np.ndarray,
    rotulos_verdadeiros: np.ndarray,
    selecionados: dict[str, ResultadoAgrupamento],
    resultado_dbscan: ResultadoAgrupamento,
) -> pd.DataFrame:
    """Avalia as partições já ajustadas no subconjunto retido pelo DBSCAN."""
    mascara = resultado_dbscan.rotulos != -1
    dados_retidos = dados[mascara]
    verdadeiros_retidos = rotulos_verdadeiros[mascara]
    dbscan_retidos = resultado_dbscan.rotulos[mascara]
    total = len(dados_retidos)
    linhas = []
    for nome, resultado in (selecionados | {"dbscan": resultado_dbscan}).items():
        rotulos = resultado.rotulos[mascara]
        linhas.append(
            {
                "algoritmo": nome,
                "grupos": len(np.unique(rotulos)),
                "amostras_avaliadas": total,
                "silhueta": (
                    calcular_silhueta(dados_retidos, rotulos) if total >= 2 else np.nan
                ),
                "ari": (
                    calcular_ari(verdadeiros_retidos, rotulos) if total >= 2 else np.nan
                ),
                "ari_com_dbscan": (
                    calcular_ari(dbscan_retidos, rotulos) if total >= 2 else np.nan
                ),
            }
        )
    return pd.DataFrame(linhas)


def _salvar_figura(fig, caminho):
    fig.savefig(caminho.with_suffix(".png"), dpi=160, bbox_inches="tight")
    fig.savefig(caminho.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


def _plotar_particionais(diretorio, tabela):
    cruzados = {
        "kernel-fuzzy-k-means": ("m", "gamma"),
        "gaussian-mixture": ("covariance_type", "reg_covar"),
    }
    with sns.axes_style("whitegrid"):
        for nome, linhas in tabela.groupby("algoritmo", sort=False):
            caminho = diretorio / f"sensibilidade_{nome.replace('-', '_')}"
            selecionadas = linhas.loc[linhas["configuracao_selecionada"]]
            if nome in cruzados:
                linha, coluna = cruzados[nome]
                selecionada = selecionadas.iloc[0]
                fig, eixos = plt.subplots(2, 2, figsize=(8, 5), layout="constrained")
                limites = (linhas["silhueta"].min(), linhas["silhueta"].max())
                for k, eixo in zip(VALORES_K, eixos.flat, strict=True):
                    matriz = linhas.loc[linhas["k"] == k].pivot(
                        index=linha, columns=coluna, values="silhueta"
                    )
                    sns.heatmap(
                        matriz,
                        ax=eixo,
                        annot=True,
                        fmt=".3f",
                        cmap="YlGnBu",
                        cbar=False,
                        vmin=limites[0],
                        vmax=limites[1],
                        linewidths=0.5,
                    )
                    y = matriz.index.get_loc(selecionada[linha])
                    x = matriz.columns.get_loc(selecionada[coluna])
                    eixo.scatter(
                        x + 0.5,
                        y + 0.5,
                        s=250,
                        marker="s",
                        facecolor="none",
                        edgecolor="#c43c39",
                        linewidth=1.5,
                    )
                    eixo.set_title(f"k = {k}")
                fig.suptitle(f"{NOMES_METODOS[nome]} · Silhueta por configuração")
            else:
                fig, eixo = plt.subplots(figsize=(6, 3.5), layout="constrained")
                chave = "m" if nome == "fuzzy-k-means" else "init"
                sns.lineplot(
                    linhas, x="k", y="silhueta", hue=chave, marker="o", ax=eixo
                )
                eixo.scatter(
                    selecionadas["k"],
                    selecionadas["silhueta"],
                    s=120,
                    facecolor="none",
                    edgecolor="#c43c39",
                    linewidth=1.5,
                    zorder=5,
                )
                eixo.set(
                    xticks=list(VALORES_K),
                    xlabel="Número de grupos (k)",
                    ylabel="Silhueta",
                    title=f"{NOMES_METODOS[nome]} · {chave}",
                )
            _salvar_figura(fig, caminho)


def _plotar_dbscan(diretorio, tabela):
    with sns.axes_style("whitegrid"):
        fig, eixos = plt.subplots(1, 2, figsize=(8, 3.6), layout="constrained")
        escolhida = tabela.loc[tabela["selecionado"]].iloc[0]
        for eixo, metrica, nome in zip(
            eixos, ("silhueta", "ari"), ("Silhueta", "ARI"), strict=True
        ):
            sns.scatterplot(
                tabela,
                x="cobertura_pct",
                y=metrica,
                hue="min_samples",
                style="regra",
                ax=eixo,
            )
            eixo.axvline(
                COBERTURA_MINIMA_DBSCAN, color="#666666", linestyle="--", linewidth=1
            )
            if np.isfinite(escolhida[metrica]):
                eixo.scatter(
                    escolhida["cobertura_pct"],
                    escolhida[metrica],
                    s=120,
                    facecolor="none",
                    edgecolor="#c43c39",
                    linewidth=1.5,
                    zorder=5,
                )
            eixo.set(xlabel="Cobertura (%)", ylabel=nome)
        fig.suptitle("DBSCAN · Métricas e cobertura")
        _salvar_figura(fig, diretorio / "sensibilidade_dbscan")


def buscar_hiperparametros(
    dados: np.ndarray,
    parametros_dbscan: tuple[float, int] | None = None,
) -> ResultadoBusca:
    """Avalia as grades piloto e escolhe uma configuração por algoritmo."""
    with threadpool_limits(limits=1):
        selecionados, registros_particionais, tabela_particionais = (
            _buscar_particionais(dados)
        )
        escolha_dbscan, registros_dbscan, tabela_dbscan = _buscar_dbscan(
            dados, parametros_dbscan
        )

    return ResultadoBusca(
        tabela_particionais,
        tabela_dbscan,
        {"particionais": registros_particionais, "dbscan": registros_dbscan},
        {
            "valores_k": list(VALORES_K),
            "sementes": list(SEMENTES_PILOTO),
            "cobertura_minima_dbscan_pct": COBERTURA_MINIMA_DBSCAN,
            "criterio_particionais": (
                "Melhor objetivo finito entre as sementes, com empate pela menor semente. "
                "Maior média aritmética das quatro silhuetas em k de 2 a 5. "
                "Empates favorecem o perfil de referência e depois a ordem da grade."
            ),
            "criterio_dbscan": (
                "Maior silhueta válida com ao menos 95% de cobertura, com empate pela "
                "maior cobertura e pela primeira configuração na grade."
            ),
            "ajustes_particionais": len(tabela_particionais) * len(SEMENTES_PILOTO),
            "ajustes_dbscan": len(tabela_dbscan),
            "selecionados": selecionados,
            "dbscan": escolha_dbscan,
            "dbscan_modo": "manual" if parametros_dbscan else "automatico",
        },
    )


def salvar_busca(
    busca: ResultadoBusca, rotulos_verdadeiros: np.ndarray, diretorio: Path
) -> None:
    """Acrescenta o ARI descritivo ao registro piloto após a rodada oficial."""
    rotulos_particionais = {
        (nome, perfil["configuracao"], k, rodada["seed"]): rodada["rotulos"]
        for nome, perfis in busca.execucoes["particionais"].items()
        for perfil in perfis
        for k, rodadas in perfil["execucoes"].items()
        for rodada in rodadas
    }
    tabela_particionais = busca.particionais.assign(
        ari=[
            (
                calcular_ari(
                    rotulos_verdadeiros,
                    rotulos_particionais[
                        (linha.algoritmo, linha.configuracao, linha.k, linha.seed)
                    ],
                )
                if pd.notna(linha.seed)
                else np.nan
            )
            for linha in busca.particionais.itertuples(index=False)
        ]
    )
    tabela_dbscan = busca.dbscan.assign(
        ari=[
            calcular_ari(rotulos_verdadeiros, rodada["rotulos"], ignorar_ruido=True)
            for rodada in busca.execucoes["dbscan"]
        ]
    )
    diretorio_figuras = diretorio / "figuras"
    diretorio_figuras.mkdir(parents=True, exist_ok=True)
    tabela_particionais.to_csv(diretorio / "particionais.csv", index=False)
    tabela_dbscan.to_csv(diretorio / "dbscan.csv", index=False)
    salvar_json(diretorio / "execucoes.json", busca.execucoes)
    _plotar_particionais(diretorio_figuras, tabela_particionais)
    _plotar_dbscan(diretorio_figuras, tabela_dbscan)




## Código completo dos experimentos

Rodadas piloto e oficial da Questão 1 e fluxo completo da Questão 2.

In [ ]:
# Fluxos completos das Questões 1 e 2.
# Algoritmos e tunagem já estão no namespace do notebook.
__file__ = str(pasta_codigo / "questoes.py")
"""Execução completa das Questões 1 e 2 do projeto AM-NS."""

import hashlib
import importlib.metadata
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE, trustworthiness
from threadpoolctl import threadpool_limits
from umap import UMAP


ROOT = Path(__file__).resolve().parent.parent
CODIGO = Path(__file__).resolve().parent
SEMENTES_OFICIAIS = range(52, 62)


def executar_questao_1(
    padronizar: bool = True,
    eps_dbscan: float | None = None,
    min_samples_dbscan: int | None = None,
    *,
    diretorio_resultados: str | Path = ROOT / "resultados",
) -> pd.DataFrame:
    """Executa a busca piloto e a rodada oficial de agrupamentos da Questão 1."""
    diretorio_resultados = Path(diretorio_resultados)
    diretorio_tunagem = diretorio_resultados / "tunagem"
    diretorio_figuras = diretorio_resultados / "figuras"
    X, y, preprocessamento = carregar_iris(padronizar=padronizar)
    valores_k = range(1, 6)
    parametros_dbscan = None
    if eps_dbscan is not None or min_samples_dbscan is not None:
        parametros_dbscan = (
            PADRAO_DBSCAN[0] if eps_dbscan is None else eps_dbscan,
            PADRAO_DBSCAN[1] if min_samples_dbscan is None else min_samples_dbscan,
        )
    print("Rodada piloto (k de 2 a 5)...")
    busca = buscar_hiperparametros(X, parametros_dbscan)

    todas_execucoes: dict[str, Any] = {}
    melhores: dict[str, dict[int, ResultadoAgrupamento]] = {}
    ajustes_particionais = 0
    print("Rodada oficial (k de 1 a 5, sementes 52–61)...")
    with threadpool_limits(limits=1):
        for nome, escolha in busca.metadados["selecionados"].items():
            parametros = escolha["parametros"]
            print(f"Executando {nome} com {parametros}...")
            execucoes = executar_todas_por_k(
                nome, X, valores_k, seed_inicial=SEMENTES_OFICIAIS.start, **parametros
            )
            ajustes_particionais += sum(len(rodadas) for rodadas in execucoes.values())
            todas_execucoes[nome] = execucoes
            melhores[nome] = selecionar_melhores_por_k(nome, execucoes)

        print("Executando upgma (uma hierarquia e cinco cortes)...")
        resultados_upgma = upgma(X, valores_k=tuple(valores_k))
        melhores["upgma"] = resultados_upgma
        todas_execucoes["upgma"] = resultados_upgma

        escolha_dbscan = busca.metadados["dbscan"]
        print(
            f"Executando dbscan com eps={escolha_dbscan['eps']:.9g} "
            f"e min_samples={escolha_dbscan['min_samples']}..."
        )
        resultado_dbscan = dbscan(
            X, eps=escolha_dbscan["eps"], min_samples=escolha_dbscan["min_samples"]
        )
    todas_execucoes["dbscan"] = resultado_dbscan

    plotar_dendrograma(resultados_upgma[2], diretorio_figuras)

    resumo = condensar_resultados(X, y, melhores, resultado_dbscan)
    diretorio_resultados.mkdir(parents=True, exist_ok=True)
    linhas = []
    contingencias = {}
    selecionados = {}
    for nome, resultados in melhores.items():
        k_estrela, silhuetas = selecionar_k_por_silhueta(X, resultados)
        plotar_silhuetas(nome, silhuetas, k_estrela, diretorio_figuras)
        selecionados[nome] = resultados[k_estrela]
        contingencias[nome] = pd.crosstab(
            pd.Series(y, name="especie"),
            pd.Series(selecionados[nome].rotulos, name="grupo"),
        ).to_dict()
        for k in valores_k:
            resultado = resultados.get(k)
            linhas.append(
                {
                    "algoritmo": nome,
                    "k": k,
                    "seed": resultado.parametros.get("seed") if resultado else None,
                    "objetivo": resultado.objetivo if resultado else None,
                    "silhueta": silhuetas.get(k),
                    "ari": calcular_ari(y, resultado.rotulos) if resultado else None,
                    "selecionado": k == k_estrela,
                }
            )
    contingencias["dbscan"] = pd.crosstab(
        pd.Series(y), pd.Series(resultado_dbscan.rotulos)
    ).to_dict()
    salvar_json(diretorio_resultados / "contingencias.json", contingencias)
    pd.DataFrame(linhas).to_csv(
        diretorio_resultados / "metricas_por_k.csv", index=False
    )
    resumo.to_csv(diretorio_resultados / "resumo_agrupamentos.csv", index=False)
    salvar_json(diretorio_resultados / "execucoes_questao_1.json", todas_execucoes)
    salvar_busca(busca, y, diretorio_tunagem)
    comparar_amostra_dbscan(X, y, selecionados, resultado_dbscan).to_csv(
        diretorio_tunagem / "comparacao_mesma_amostra.csv", index=False
    )
    rodada_oficial = {
        "sementes": list(SEMENTES_OFICIAIS),
        "ajustes_particionais": ajustes_particionais,
        "ajustes_upgma": 1,
        "ajustes_dbscan": 1,
        "ajustes_totais": ajustes_particionais + 2,
        "criterio_k": "Maior silhueta válida, com empate pelo menor k.",
    }
    salvar_json(
        diretorio_resultados / "metadados_questao_1.json",
        {
            "base": "codigo/iris/iris.data (UCI Iris)",
            "preprocessamento": preprocessamento,
            "valores_k": list(valores_k),
            "execucoes_por_algoritmo_particional": len(SEMENTES_OFICIAIS),
            "silhueta": {"metrica": "euclidean", "k_1": "indefinida (NaN)"},
            "dbscan": {
                "eps": resultado_dbscan.parametros["eps"],
                "min_samples": resultado_dbscan.parametros["min_samples"],
                "silhueta": "pontos de ruído (-1) removidos",
                "ari": "pontos de ruído (-1) removidos",
            },
            "rodada_oficial": rodada_oficial,
            "tunagem": busca.metadados
            | {
                "ajustes_totais": (
                    busca.metadados["ajustes_particionais"]
                    + busca.metadados["ajustes_dbscan"]
                    + rodada_oficial["ajustes_totais"]
                ),
            },
            "versoes": {
                "python": sys.version,
                **{
                    pacote: importlib.metadata.version(pacote)
                    for pacote in [
                        "numpy",
                        "pandas",
                        "scikit-learn",
                        "scipy",
                        "matplotlib",
                        "seaborn",
                        "pydantic",
                        "fuzzy-c-means",
                        "kmedoids",
                    ]
                },
            },
        },
    )
    print("\nResumo dos agrupamentos:")
    print(resumo.to_string(index=False, float_format=lambda valor: f"{valor:.4f}"))
    print(f"\nResultados salvos em: {diretorio_resultados}")
    return resumo


def executar_questao_2(
    padronizar: bool = True,
    seed: int = 42,
    gamma: float = 0.25,
    perplexidade: float = 30.0,
    vizinhos: int = 15,
    min_dist: float = 0.1,
) -> dict[str, Any]:
    """Executa PCA, Kernel PCA, t-SNE e UMAP da Questão 2."""
    diretorio_resultados = ROOT / "resultados"
    diretorio_figuras = ROOT / "resultados" / "figuras"
    diretorio_resultados.mkdir(exist_ok=True)
    diretorio_figuras.mkdir(exist_ok=True)
    X, y, prep = carregar_iris(padronizar=padronizar)
    modelos = {
        "pca": PCA(n_components=4, svd_solver="full"),
        "kernel_pca": KernelPCA(
            n_components=2, kernel="rbf", gamma=gamma, eigen_solver="dense"
        ),
        "tsne": TSNE(
            n_components=2,
            perplexity=perplexidade,
            learning_rate="auto",
            init="pca",
            max_iter=1000,
            random_state=seed,
        ),
        "umap": UMAP(
            n_components=2,
            n_neighbors=vizinhos,
            min_dist=min_dist,
            metric="euclidean",
            random_state=seed,
            n_jobs=1,
        ),
    }
    resultados = {}
    for nome, modelo in modelos.items():
        print(f"Executando {nome}...", flush=True)
        Z = modelo.fit_transform(X)
        tabela = pd.DataFrame(
            Z, columns=[f"componente_{i+1}" for i in range(Z.shape[1])]
        )
        tabela.insert(0, "observacao", np.arange(1, len(X) + 1))
        tabela["especie"] = y
        tabela.to_csv(diretorio_resultados / f"coordenadas_{nome}.csv", index=False)
        resultado = {
            "parametros": modelo.get_params(),
            "coordenadas": Z,
            "trustworthiness_10": trustworthiness(X, Z[:, :2], n_neighbors=10),
        }
        if nome == "pca":
            resultado.update(
                componentes=modelo.components_,
                autovalores=modelo.explained_variance_,
                variancia_explicada=modelo.explained_variance_ratio_,
                variancia_acumulada=np.cumsum(modelo.explained_variance_ratio_),
                valores_singulares=modelo.singular_values_,
                media=modelo.mean_,
                loadings=modelo.components_.T * np.sqrt(modelo.explained_variance_),
                convencao_loadings="autovetores multiplicados pela raiz do autovalor",
            )
            pd.DataFrame(
                modelo.components_.T,
                index=prep["atributos"],
                columns=["PC1", "PC2", "PC3", "PC4"],
            ).to_csv(diretorio_resultados / "pca_componentes.csv")
            pd.DataFrame(
                resultado["loadings"],
                index=prep["atributos"],
                columns=["PC1", "PC2", "PC3", "PC4"],
            ).to_csv(diretorio_resultados / "pca_loadings.csv")
        elif nome == "kernel_pca":
            resultado.update(
                autovalores=modelo.eigenvalues_, autovetores=modelo.eigenvectors_
            )
        elif nome == "tsne":
            resultado.update(
                divergencia_kl=modelo.kl_divergence_,
                iteracoes=modelo.n_iter_,
                taxa_aprendizado_efetiva=modelo.learning_rate_,
            )
        resultados[nome] = resultado
        sns.set_theme(style="whitegrid")
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=y, style=y, s=48, ax=ax)
        ax.set(
            title=(
                nome.upper().replace("_", " ") + " — Iris padronizada"
                if padronizar
                else nome.upper()
            ),
            xlabel="Componente 1",
            ylabel="Componente 2",
        )
        if nome == "pca":
            ax.set_xlabel(f"PC1 ({modelo.explained_variance_ratio_[0]:.2%})")
            ax.set_ylabel(f"PC2 ({modelo.explained_variance_ratio_[1]:.2%})")
        ax.legend(title="Espécie", fontsize=8)
        fig.tight_layout()
        fig.savefig(diretorio_figuras / f"{nome}.png", dpi=170)
        plt.close(fig)
    salvar_json(diretorio_resultados / "execucoes_questao_2.json", resultados)
    salvar_json(
        diretorio_resultados / "metadados_questao_2.json",
        {
            "preprocessamento": prep,
            "seed": seed,
            "python": sys.version,
            "fonte_requisitos": "Projeto-AMG-NS-2026-2.pdf; Equipe 08",
            "sha256_dados": hashlib.sha256(
                (CODIGO / "iris/iris.data").read_bytes()
            ).hexdigest(),
            "versoes": {
                pacote: importlib.metadata.version(pacote)
                for pacote in [
                    "numpy",
                    "pandas",
                    "scikit-learn",
                    "scipy",
                    "umap-learn",
                    "numba",
                    "matplotlib",
                ]
            },
        },
    )
    return resultados


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE, trustworthiness
from umap import UMAP


X, y, pre = carregar_iris()
ATRIBUTOS = pre["atributos"]
GAMMA_KERNEL = 0.25
PERPLEXIDADE, VIZINHOS_UMAP, MIN_DIST_UMAP, SEED_Q2 = 30, 15, 0.1, 42
sns.set_theme(style="whitegrid")

## Questão 1 — Agrupamento

No piloto, cada configuração particional usa as sementes 42–51 em `k=2..5`. Em cada `k`,
a melhor execução pela função objetivo fornece a silhueta. A média das quatro silhuetas
define a configuração, com prioridade para a referência em caso de empate.

Na rodada oficial, a configuração fica fixa e cada `k=1..5` recebe dez novas inicializações
com as sementes 52–61. A função objetivo decide a melhor execução por `k`. A maior
silhueta válida define `k*`, seguida da avaliação por ARI. A silhueta em `k=1` é indefinida.
O UPGMA usa uma árvore Manhattan de ligação média para os cinco cortes.

O DBSCAN seleciona pela silhueta com cobertura mínima de 95% no piloto. O par escolhido
é executado uma vez na rodada oficial. Sua silhueta e seu ARI excluem o ruído.

In [ ]:
pasta_saida = TemporaryDirectory(prefix="amns-q1-")
resultados_q1 = Path(pasta_saida.name)
resumo = executar_questao_1(diretorio_resultados=resultados_q1)
display(resumo.round(4))

### Número de grupos, dendrograma e comparação na mesma amostra

In [ ]:
metricas = pd.read_csv(resultados_q1 / "metricas_por_k.csv")
for metrica in ("silhueta", "ari"):
    display(metricas.pivot(index="algoritmo", columns="k", values=metrica).round(4))
for figura in sorted((resultados_q1 / "figuras").glob("*.png")):
    display(Image(filename=str(figura), width=650))
comparacao = pd.read_csv(resultados_q1 / "tunagem" / "comparacao_mesma_amostra.csv")
display(comparacao.round(4))

### Sensibilidade aos hiperparâmetros

As tabelas mostram as configurações avaliadas. Na leitura do DBSCAN, acompanhe a
silhueta e o ARI com a cobertura. Nas curvas particionais, compare cada configuração
ao longo dos valores de `k`.

In [ ]:
tunagem = resultados_q1 / "tunagem"
for tabela in ("particionais", "dbscan"):
    dados = pd.read_csv(tunagem / f"{tabela}.csv")
    display(dados.round(4))
for figura in sorted((tunagem / "figuras").glob("*.png")):
    display(Image(filename=str(figura), width=750))

## Questão 2 — PCA, Kernel PCA, t-SNE e UMAP
Todas as técnicas usam a mesma matriz padronizada com os 4 atributos. Para cada uma mostramos as saídas, o gráfico 2D e a trustworthiness (quanto dos 10 vizinhos mais próximos foi preservado, de 0 a 1).

In [ ]:
def scatter2d(Z, titulo, xl="Componente 1", yl="Componente 2"):
    fig, ax = plt.subplots(figsize=(8, 5.5))
    sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=y, style=y, s=55, ax=ax)
    ax.set(title=titulo, xlabel=xl, ylabel=yl)
    ax.legend(title="Espécie", fontsize=8)
    imagem = io.BytesIO()
    fig.savefig(imagem, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    display(Image(data=imagem.getvalue()))

trust = {}

### 1 PCA

In [ ]:
pca = PCA(n_components=4, svd_solver="full")
Z_pca = pca.fit_transform(X)
cp = [f"PC{i+1}" for i in range(4)]

saida_pca = pd.DataFrame({"Autovalor": pca.explained_variance_,
                          "Variância (%)": 100 * pca.explained_variance_ratio_,
                          "Acumulada (%)": 100 * np.cumsum(pca.explained_variance_ratio_),
                          "Valor singular": pca.singular_values_}, index=cp)
display(saida_pca.round(4))

print("Componentes (autovetores):")
display(pd.DataFrame(pca.components_.T, index=ATRIBUTOS, columns=cp).round(4))

print("Loadings (autovetor x raiz do autovalor):")
display(pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_), index=ATRIBUTOS, columns=cp).round(4))

trust["PCA"] = trustworthiness(X, Z_pca[:, :2], n_neighbors=10)
scatter2d(Z_pca, "PCA - Iris padronizada",
          f"PC1 ({pca.explained_variance_ratio_[0]:.2%})", f"PC2 ({pca.explained_variance_ratio_[1]:.2%})")

### 2 Kernel PCA (RBF)

In [ ]:
kpca = KernelPCA(n_components=2, kernel="rbf", gamma=GAMMA_KERNEL, eigen_solver="dense")
Z_kpca = kpca.fit_transform(X)
print("Autovalores do kernel centralizado:", np.round(kpca.eigenvalues_, 4))
print("Parâmetros:", kpca.get_params())
trust["Kernel PCA"] = trustworthiness(X, Z_kpca, n_neighbors=10)
scatter2d(Z_kpca, "Kernel PCA (RBF) - Iris padronizada")

### 3 t-SNE

In [ ]:
tsne = TSNE(n_components=2, perplexity=PERPLEXIDADE, learning_rate="auto", init="pca",
            max_iter=1000, random_state=SEED_Q2)
Z_tsne = tsne.fit_transform(X)
print("Divergência KL final:", tsne.kl_divergence_)
print("Iterações:", tsne.n_iter_, "| taxa de aprendizado efetiva:", tsne.learning_rate_)
trust["t-SNE"] = trustworthiness(X, Z_tsne, n_neighbors=10)
scatter2d(Z_tsne, "t-SNE - Iris padronizada")

### 4 UMAP

In [ ]:
umap = UMAP(n_components=2, n_neighbors=VIZINHOS_UMAP, min_dist=MIN_DIST_UMAP,
            metric="euclidean", random_state=SEED_Q2, n_jobs=1)
Z_umap = umap.fit_transform(X)
print("Parâmetros:", dict(n_neighbors=VIZINHOS_UMAP, min_dist=MIN_DIST_UMAP, metric="euclidean"))
trust["UMAP"] = trustworthiness(X, Z_umap, n_neighbors=10)
scatter2d(Z_umap, "UMAP - Iris padronizada")

### 5 Comparação das técnicas

In [ ]:
display(pd.Series(trust, name="Trustworthiness (10 vizinhos)").round(4).to_frame())